## Build `author_observed_orcids`: every trusted ORCID on each author's seats

Feeds `CreateAuthors` (`observed_orcids` array, and the organic primary `orcid`). oxjob #1267.

**A seat ORCID is trusted when** (the #1262 filter, SQL stages; the desk-side v2 name-rescue pass is not run here, so this errs toward dropping):
1. it is the only seat on its work carrying that ORCID (publishers stamp one ORCID on every authorship);
2. the work is not a data deposit (`provenance <> 'datacite'`, `type <> 'dataset'`) and not `mag` provenance (mag `raw_orcid` is the legacy profile's ORCID copied back onto the seat: circular);
3. the seat's parsed surname is compatible with the surname the ORCID carries on most of its seats, and the first name does not conflict outright;
4. the ORCID is not a multi-family ORCID (>= 5 seats, majority surname < 80%).

Output `openalex.authors.author_observed_orcids`: one row per profile with `seat_orcids` = `ARRAY<STRUCT<orcid, n_seats, last_year>>` sorted most-seats-first. `CreateAuthors` applies curations (removed ORCID dropped, curated primary prepended) and derives `orcid` + `observed_orcids`.

**Invariant** (reported, not enforced): an ORCID belongs to at most one profile. Collisions are the splinter profiles that #1263's ORCID-collision merges clean up; the count is written to `author_observed_orcid_collisions` and should only shrink.

In [ ]:
%sql
-- 1. Every assigned seat that carries a raw ORCID, with the signals the filter needs.
CREATE OR REPLACE TABLE openalex.authors.author_orcid_seats
CLUSTER BY (raw_orcid) AS
WITH seats AS (
  SELECT w.id AS work_id,
         a.author_order_number AS author_sequence,
         CAST(REPLACE(a.author.id, 'https://openalex.org/A', '') AS BIGINT) AS author_id,
         TRIM(a.raw_author_name) AS raw_author_name,
         a.raw_orcid,
         w.primary_location.provenance AS provenance,
         w.type AS work_type,
         COALESCE(w.publication_year, YEAR(w.publication_date)) AS publication_year
  FROM openalex.works.openalex_works w
  LATERAL VIEW explode(FILTER(w.authorships, x -> x.raw_orcid IS NOT NULL)) t AS a
  WHERE w.authorships IS NOT NULL
    AND a.author.id IS NOT NULL
    AND a.author.id NOT IN ('https://openalex.org/A9999999999', 'https://openalex.org/A5317838346',
                            'https://openalex.org/A5098778687', 'https://openalex.org/A5040317105')
)
SELECT s.*,
       COUNT(*) OVER (PARTITION BY s.work_id, s.raw_orcid) AS orcid_seats_on_work,
       an.match_first,
       an.match_last
FROM seats s
LEFT JOIN openalex.authors.author_names an ON an.raw_author_name = s.raw_author_name;

In [ ]:
%sql
-- 2. Canonical name per ORCID: the surname it carries on most of its non-deposit, non-mag seats, and the most common full first name there.
CREATE OR REPLACE TABLE openalex.authors.author_orcid_canon AS
WITH f AS (
  SELECT raw_orcid, match_last, match_first
  FROM openalex.authors.author_orcid_seats
  WHERE orcid_seats_on_work = 1
    AND provenance NOT IN ('datacite', 'mag') AND COALESCE(work_type, '') <> 'dataset'
    AND match_last IS NOT NULL
),
per_last AS (SELECT raw_orcid, match_last, COUNT(*) AS n FROM f GROUP BY 1, 2),
canon AS (
  SELECT raw_orcid,
         MAX_BY(match_last, n) AS canon_last,
         SUM(n) AS n_seats,
         COUNT(*) AS n_distinct_last,
         MAX(n) / SUM(n) AS canon_last_share
  FROM per_last GROUP BY 1
),
per_first AS (
  SELECT f.raw_orcid, f.match_first, COUNT(*) AS n
  FROM f JOIN canon c ON c.raw_orcid = f.raw_orcid AND f.match_last = c.canon_last
  WHERE f.match_first IS NOT NULL AND LENGTH(f.match_first) > 1
  GROUP BY 1, 2
)
SELECT c.*, pf.canon_first
FROM canon c
LEFT JOIN (SELECT raw_orcid, MAX_BY(match_first, n) AS canon_first FROM per_first GROUP BY 1) pf USING (raw_orcid);

In [ ]:
%sql
-- 3. Trusted seats: apply the filter. Reason codes kept for the diagnostics cell; NULL reason = trusted.
CREATE OR REPLACE TABLE openalex.authors.author_orcid_seats_trusted
CLUSTER BY (author_id) AS
WITH j AS (
  SELECT s.*, c.canon_last, c.canon_first, c.n_seats AS orcid_n_seats, c.n_distinct_last, c.canon_last_share,
         (s.match_last = c.canon_last OR INSTR(s.match_last, c.canon_last) > 0 OR INSTR(c.canon_last, s.match_last) > 0) AS last_ok,
         (s.match_first IS NULL OR c.canon_first IS NULL
            OR LENGTH(s.match_first) = 1 OR LENGTH(c.canon_first) = 1
            OR s.match_first = c.canon_first
            OR INSTR(s.match_first, c.canon_first) > 0 OR INSTR(c.canon_first, s.match_first) > 0) AS first_ok,
         (SUBSTRING(s.match_first, 1, 1) <=> SUBSTRING(c.canon_first, 1, 1)) AS initial_ok
  FROM openalex.authors.author_orcid_seats s
  LEFT JOIN openalex.authors.author_orcid_canon c ON c.raw_orcid = s.raw_orcid
),
flagged AS (
  SELECT j.*,
    CASE
      WHEN j.orcid_seats_on_work > 1 THEN 'dup_on_work'
      WHEN j.provenance = 'datacite' OR COALESCE(j.work_type, '') = 'dataset' THEN 'deposit'
      WHEN j.provenance = 'mag' THEN 'provenance_mag'
      WHEN j.match_last IS NULL THEN 'unparsed'
      WHEN j.canon_last IS NULL THEN 'no_canon'
      WHEN NOT j.last_ok THEN 'surname_mismatch'
      WHEN NOT j.first_ok AND NOT j.initial_ok THEN 'first_conflict'
      WHEN j.n_distinct_last >= 2 AND j.canon_last_share < 0.8 AND j.orcid_n_seats >= 5 THEN 'orcid_multi_family'
    END AS reason
  FROM j
)
SELECT work_id, author_sequence, author_id, raw_orcid, provenance, work_type, publication_year, reason
FROM flagged;

In [ ]:
%sql
-- 4. Per-profile array of trusted ORCIDs, most seats first, then most recent.
CREATE OR REPLACE TABLE openalex.authors.author_observed_orcids
CLUSTER BY (author_id) AS
WITH per_orcid AS (
  SELECT author_id, raw_orcid AS orcid, COUNT(*) AS n_seats, MAX(publication_year) AS last_year
  FROM openalex.authors.author_orcid_seats_trusted
  WHERE reason IS NULL
  GROUP BY 1, 2
)
SELECT author_id,
       TRANSFORM(
         ARRAY_SORT(COLLECT_LIST(STRUCT(-n_seats AS neg_n, -COALESCE(last_year, 0) AS neg_year, orcid, n_seats, last_year))),
         x -> STRUCT(x.orcid AS orcid, CAST(x.n_seats AS INT) AS n_seats, x.last_year AS last_year)
       ) AS seat_orcids,
       CAST(SUM(n_seats) AS INT) AS orcid_seats
FROM per_orcid
GROUP BY author_id;

OPTIMIZE openalex.authors.author_observed_orcids;

In [ ]:
%sql
-- 5. Diagnostics: filter removal counts and the one-ORCID-one-profile invariant (#1267 acceptance test 3).
CREATE OR REPLACE TABLE openalex.authors.author_observed_orcid_collisions AS
SELECT orcid, COLLECT_SET(author_id) AS author_ids, COUNT(DISTINCT author_id) AS n_profiles
FROM (SELECT author_id, s.orcid FROM openalex.authors.author_observed_orcids LATERAL VIEW EXPLODE(seat_orcids) e AS s)
GROUP BY orcid HAVING COUNT(DISTINCT author_id) >= 2;

SELECT 'removed:' || COALESCE(reason, 'KEPT') AS bucket, COUNT(*) AS seats, COUNT(DISTINCT raw_orcid) AS orcids
FROM openalex.authors.author_orcid_seats_trusted GROUP BY 1
UNION ALL
SELECT 'profiles_with_observed', COUNT(*), NULL FROM openalex.authors.author_observed_orcids
UNION ALL
SELECT 'profiles_with_2plus', COUNT(*), NULL FROM openalex.authors.author_observed_orcids WHERE SIZE(seat_orcids) >= 2
UNION ALL
SELECT 'invariant_violations_orcids', COUNT(*), NULL FROM openalex.authors.author_observed_orcid_collisions
ORDER BY 1;